# Test Metrics Summary

This notebook reads a `test_metrics_over_all.csv` file and prints mean/std per metric, overall and by body part.
Use the utility functions below to load a CSV and display results for multiple models in the same notebook.


In [13]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

BASE_METRICS = ["MAE", "MSE", "PSNR", "SSIM"]


def _metric_columns(base_metrics=BASE_METRICS):
    unmasked = [f"{m}_unmasked" for m in base_metrics]
    masked = [f"{m}_masked" for m in base_metrics]
    return unmasked + masked


def _sanitize_metric_columns(df: pd.DataFrame, metrics) -> pd.DataFrame:
    """Convert +/-inf to NaN so mean/std are computed on finite values only."""
    df = df.copy()
    for col in metrics:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df[col] = df[col].replace([np.inf, -np.inf], np.nan)
    return df


def _ensure_metric_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize metrics to *_unmasked and *_masked columns."""
    df = df.copy()
    suffixed_cols = _metric_columns()
    if all(col in df.columns for col in suffixed_cols):
        return df

    # Backward compatibility with old CSVs that only had MAE/MSE/PSNR/SSIM.
    if all(col in df.columns for col in BASE_METRICS):
        for metric in BASE_METRICS:
            df[f"{metric}_unmasked"] = pd.to_numeric(df[metric], errors="coerce")
            df[f"{metric}_masked"] = pd.to_numeric(df[metric], errors="coerce")
        return df

    missing = [c for c in suffixed_cols if c not in df.columns]
    raise ValueError(f"Could not find expected metric columns. Missing: {missing}")


def load_metrics_csv(csv_path: Path) -> pd.DataFrame:
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV not found: {csv_path}")

    df = pd.read_csv(csv_path)

    if "file_name" not in df.columns:
        df = df.iloc[:, 1:]

    if "file_name" not in df.columns:
        raise ValueError("Could not find 'file_name' column after dropping index column.")

    df = _ensure_metric_columns(df)
    df = _sanitize_metric_columns(df, _metric_columns())

    if "mask_voxels" in df.columns:
        df["mask_voxels"] = pd.to_numeric(df["mask_voxels"], errors="coerce")

    return df


def load_volume_metrics_csv(volume_csv_path: Path) -> pd.DataFrame:
    if not volume_csv_path.exists():
        raise FileNotFoundError(f"CSV not found: {volume_csv_path}")

    df = pd.read_csv(volume_csv_path)

    if "volume_id" not in df.columns:
        df = df.iloc[:, 1:]

    if "volume_id" not in df.columns:
        raise ValueError("Could not find 'volume_id' column after dropping index column.")

    df = _ensure_metric_columns(df)
    df = _sanitize_metric_columns(df, _metric_columns())

    if "mask_voxels" in df.columns:
        df["mask_voxels"] = pd.to_numeric(df["mask_voxels"], errors="coerce")
    if "slice_count" in df.columns:
        df["slice_count"] = pd.to_numeric(df["slice_count"], errors="coerce")

    return df


def _fmt(v: float) -> str:
    return "n/a" if pd.isna(v) else f"{v:.6g}"


def overall_mean_std(df: pd.DataFrame, metrics) -> pd.DataFrame:
    overall_mean = df[metrics].mean()
    overall_std = df[metrics].std()
    return pd.DataFrame({"mean +- std": [f"{_fmt(m)} +- {_fmt(s)}" for m, s in zip(overall_mean, overall_std)]}, index=metrics)


def paired_mean_std(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for metric in BASE_METRICS:
        u_col = f"{metric}_unmasked"
        m_col = f"{metric}_masked"
        u_mean, u_std = df[u_col].mean(), df[u_col].std()
        m_mean, m_std = df[m_col].mean(), df[m_col].std()
        rows.append({
            "metric": metric,
            "unmasked (mean +- std)": f"{_fmt(u_mean)} +- {_fmt(u_std)}",
            "masked (mean +- std)": f"{_fmt(m_mean)} +- {_fmt(m_std)}",
        })
    return pd.DataFrame(rows).set_index("metric")


def add_body_part(df: pd.DataFrame, col: str = "file_name") -> pd.DataFrame:
    # Body part grouping derived from filename prefix (e.g., AB_*, HN_*)
    df = df.copy()
    df["body_part"] = df[col].astype(str).str.split("_").str[0]
    return df


def mean_std_by_body_part(df: pd.DataFrame) -> pd.DataFrame:
    grouped = df.groupby("body_part")[_metric_columns()].agg(["mean", "std"])

    formatted = pd.DataFrame(index=grouped.index)
    for metric in BASE_METRICS:
        for suffix in ["unmasked", "masked"]:
            col = f"{metric}_{suffix}"
            mean_col = (col, "mean")
            std_col = (col, "std")
            formatted[col] = grouped.apply(
                lambda row: f"{_fmt(row[mean_col])} +- {_fmt(row[std_col])}",
                axis=1,
            )
    return formatted


def _print_non_finite_counts(df: pd.DataFrame, title: str) -> None:
    issues = {}
    for metric in _metric_columns():
        non_finite = (~np.isfinite(df[metric])).sum()
        if non_finite:
            issues[metric] = int(non_finite)
    if issues:
        counts = ", ".join([f"{m}: {c}" for m, c in issues.items()])
        print(f"{title} | non-finite values ignored in stats -> {counts}")


def show_metrics_for_csv(csv_path: Path) -> None:
    df = load_metrics_csv(csv_path)

    _print_non_finite_counts(df, "Slice metrics")
    print("Overall metrics (slice-based):")
    display(paired_mean_std(df))

    df_bp = add_body_part(df, col="file_name")
    print("Metrics by body part (slice-based):")
    display(mean_std_by_body_part(df_bp))

    volume_csv_path = csv_path.parent / "test_metrics_over_volume.csv"
    if volume_csv_path.exists():
        vol_df = load_volume_metrics_csv(volume_csv_path)
        _print_non_finite_counts(vol_df, "Volume metrics")
        print("Overall metrics (volume-based):")
        display(paired_mean_std(vol_df))

        vol_df_bp = add_body_part(vol_df, col="volume_id")
        print("Metrics by body part (volume-based):")
        display(mean_std_by_body_part(vol_df_bp))
    else:
        print(f"Volume metrics CSV not found: {volume_csv_path}")
        print("Run: python preprocessing/preprocessing_synthrad/110compute_volume_metrics.py")



In [14]:
# Per-slice metrics and influence for a single volume

def show_volume_slice_influence(df: pd.DataFrame, volume=None) -> None:
    # volume can be a volume_id string (e.g., 'AB_1ABA009') or an index into the unique volumes list
    if "file_name" not in df.columns:
        raise ValueError("df must contain a 'file_name' column.")

    def _volume_key(name: str) -> str:
        base = name
        if base.endswith(".nii.gz"):
            base = base[:-7]
        elif base.endswith(".nii"):
            base = base[:-4]
        # Use the part before the last '-' as volume id
        return base.rsplit("-", 1)[0]

    def _slice_key(name: str) -> str:
        base = name
        if base.endswith(".nii.gz"):
            base = base[:-7]
        elif base.endswith(".nii"):
            base = base[:-4]
        parts = base.rsplit("-", 1)
        return parts[1] if len(parts) == 2 else ""

    df = _ensure_metric_columns(df.copy())
    df["volume_id"] = df["file_name"].astype(str).map(_volume_key)
    df["slice_id"] = df["file_name"].astype(str).map(_slice_key)

    volumes = sorted(df["volume_id"].dropna().unique())
    if not volumes:
        print("No volumes found in df.")
        return

    if volume is None:
        volume_id = volumes[0]
    elif isinstance(volume, int):
        if volume < 0 or volume >= len(volumes):
            raise IndexError(f"volume index out of range: {volume} (0..{len(volumes)-1})")
        volume_id = volumes[volume]
    else:
        volume_id = str(volume)
        if volume_id not in volumes:
            raise ValueError(f"volume_id not found: {volume_id}")

    vol_df = df[df["volume_id"] == volume_id].copy()
    if vol_df.empty:
        print(f"No slices found for volume {volume_id}.")
        return

    metrics_cols = _metric_columns()
    vol_df[metrics_cols] = vol_df[metrics_cols].apply(pd.to_numeric, errors="coerce")

    if "mask_voxels" not in vol_df.columns:
        print("No mask_voxels column found; cannot compute masked slice influence.")
        display(vol_df[["file_name", "slice_id", *metrics_cols]])
        return

    vol_df["mask_voxels"] = pd.to_numeric(vol_df["mask_voxels"], errors="coerce").fillna(0)
    total_mask = vol_df["mask_voxels"].sum()
    if total_mask == 0:
        print("mask_voxels sum to 0 for this volume; cannot compute masked influence.")
        display(vol_df[["file_name", "slice_id", *metrics_cols, "mask_voxels"]])
        return

    vol_df["weight_pct"] = (vol_df["mask_voxels"] / total_mask) * 100

    # Volume-weighted masked metrics + unmasked means for this volume
    summary_rows = []
    for metric in BASE_METRICS:
        u_col = f"{metric}_unmasked"
        m_col = f"{metric}_masked"
        weights = vol_df["mask_voxels"] / total_mask
        summary_rows.append({
            "metric": metric,
            "unmasked_mean": vol_df[u_col].mean(),
            "masked_weighted_mean": (vol_df[m_col] * weights).sum(),
        })

    print(f"Volume: {volume_id} (slices={len(vol_df)}, total_mask={int(total_mask)})")
    display(pd.DataFrame(summary_rows).set_index("metric"))

    # Per-slice influence table
    show_cols = ["slice_id", *metrics_cols, "mask_voxels", "weight_pct", "file_name"]
    display(vol_df.sort_values("slice_id")[show_cols])

# Example usage (pick a volume index or id)
# show_volume_slice_influence(df, volume=0)
# show_volume_slice_influence(df, volume="AB_1ABA009")



# CUT


## Abdomen


In [15]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_abdomen_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [16]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_abdomen_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,35.948 +- 21.1952,116.803 +- 59.6723
MSE,1308.45 +- 526.058,4279.36 +- 1541.6
PSNR,26.8588 +- 4.29121,21.9252 +- 5.66777
SSIM,0.836466 +- 0.0492183,0.621217 +- 0.071976


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,35.948 +- 21.1952,116.803 +- 59.6723,1308.45 +- 526.058,4279.36 +- 1541.6,26.8588 +- 4.29121,21.9252 +- 5.66777,0.836466 +- 0.0492183,0.621217 +- 0.071976


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,35.588 +- 9.69072,119.202 +- 29.6626
MSE,1346.45 +- 361.276,4518.63 +- 844.638
PSNR,27.067 +- 1.57333,21.3842 +- 1.76962
SSIM,0.835786 +- 0.0277656,0.619675 +- 0.0309146


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,35.588 +- 9.69072,119.202 +- 29.6626,1346.45 +- 361.276,4518.63 +- 844.638,27.067 +- 1.57333,21.3842 +- 1.76962,0.835786 +- 0.0277656,0.619675 +- 0.0309146


## Brain


In [17]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_brain_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [18]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_brain_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,45.1682 +- 37.769,180.717 +- 149.697
MSE,474.589 +- 312.748,1638.68 +- 839.773
PSNR,23.5588 +- 3.15695,17.8059 +- 3.67163
SSIM,0.898355 +- 0.0516616,0.73428 +- 0.156331


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
brain,45.1682 +- 37.769,180.717 +- 149.697,474.589 +- 312.748,1638.68 +- 839.773,23.5588 +- 3.15695,17.8059 +- 3.67163,0.898355 +- 0.0516616,0.73428 +- 0.156331


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,44.9316 +- 10.336,150.888 +- 28.8315
MSE,472.83 +- 67.7782,1566.75 +- 151.688
PSNR,23.5911 +- 1.08483,18.5715 +- 0.988191
SSIM,0.898735 +- 0.0165647,0.756722 +- 0.0374324


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
brain,44.9316 +- 10.336,150.888 +- 28.8315,472.83 +- 67.7782,1566.75 +- 151.688,23.5911 +- 1.08483,18.5715 +- 0.988191,0.898735 +- 0.0165647,0.756722 +- 0.0374324


## Pelvis


In [19]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_pelvis_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [20]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_pelvis_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,29.4105 +- 43.234,97.3549 +- 66.2945
MSE,668.096 +- 331.529,2447.62 +- 486.721
PSNR,27.1093 +- 2.55843,21.6155 +- 2.54583
SSIM,0.890386 +- 0.0467566,0.750303 +- 0.0881869


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
pelvis,29.4105 +- 43.234,97.3549 +- 66.2945,668.096 +- 331.529,2447.62 +- 486.721,27.1093 +- 2.55843,21.6155 +- 2.54583,0.890386 +- 0.0467566,0.750303 +- 0.0881869


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,29.0182 +- 7.61076,95.9329 +- 16.9005
MSE,664.909 +- 104.646,2421.15 +- 277.17
PSNR,27.1088 +- 1.12139,21.6941 +- 1.18487
SSIM,0.891334 +- 0.0162652,0.750814 +- 0.0344813


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
pelvis,29.0182 +- 7.61076,95.9329 +- 16.9005,664.909 +- 104.646,2421.15 +- 277.17,27.1088 +- 1.12139,21.6941 +- 1.18487,0.891334 +- 0.0162652,0.750814 +- 0.0344813


## Thorax


In [21]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_TH_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [22]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_TH_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,138.972 +- 402.622,234.328 +- 378.003
MSE,-887.216 +- 7465.65,2845.48 +- 6353.18
PSNR,24.1029 +- 6.05259,18.6195 +- 4.80353
SSIM,0.80573 +- 0.206773,0.624402 +- 0.172316


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
TH,138.972 +- 402.622,234.328 +- 378.003,-887.216 +- 7465.65,2845.48 +- 6353.18,24.1029 +- 6.05259,18.6195 +- 4.80353,0.80573 +- 0.206773,0.624402 +- 0.172316


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,143.767 +- 91.2981,155.654 +- 38.3489
MSE,-961.782 +- 1650.82,3709.92 +- 614.935
PSNR,24.0211 +- 2.28105,19.5478 +- 1.77913
SSIM,0.802674 +- 0.0568185,0.657615 +- 0.0486513


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
TH,143.767 +- 91.2981,155.654 +- 38.3489,-961.782 +- 1650.82,3709.92 +- 614.935,24.0211 +- 2.28105,19.5478 +- 1.77913,0.802674 +- 0.0568185,0.657615 +- 0.0486513


## Head and Neck


In [23]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_HN_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [24]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_HN_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,236.363 +- 540.441,327.994 +- 510.283
MSE,813.281 +- 2596.32,1172.71 +- 7577.41
PSNR,20.3596 +- 6.61241,16.6888 +- 5.50786
SSIM,0.728183 +- 0.227576,0.584582 +- 0.198495


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
HN,236.363 +- 540.441,327.994 +- 510.283,813.281 +- 2596.32,1172.71 +- 7577.41,20.3596 +- 6.61241,16.6888 +- 5.50786,0.728183 +- 0.227576,0.584582 +- 0.198495


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,275.379 +- 164.623,310.551 +- 165.448
MSE,724.645 +- 442.749,896.636 +- 1543.59
PSNR,19.6525 +- 3.22954,16.6319 +- 3.02403
SSIM,0.712673 +- 0.0714655,0.582894 +- 0.0933291


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
HN,275.379 +- 164.623,310.551 +- 165.448,724.645 +- 442.749,896.636 +- 1543.59,19.6525 +- 3.22954,16.6319 +- 3.02403,0.712673 +- 0.0714655,0.582894 +- 0.0933291


## Fullbody


In [25]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_allregions_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [26]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_allregions_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,109.811 +- 360.05,205.085 +- 349.063
MSE,951.012 +- 1220.74,3197.04 +- 3920.73
PSNR,24.099 +- 5.43733,18.8944 +- 4.66556
SSIM,0.836686 +- 0.165354,0.668309 +- 0.16569


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,136.385 +- 437.124,208.565 +- 413.321,1297.92 +- 1033.07,4593.66 +- 4519.29,25.0029 +- 6.08344,20.0178 +- 4.96297,0.808487 +- 0.188536,0.604662 +- 0.149681
HN,228.558 +- 537.812,310.068 +- 511.107,1629.15 +- 2432.19,4027.62 +- 6037.01,21.1369 +- 6.91356,17.4662 +- 5.78229,0.745838 +- 0.234248,0.619647 +- 0.210851
TH,156.333 +- 475.492,248.087 +- 451.235,973.155 +- 678.099,4153.63 +- 4745.34,24.1695 +- 6.39041,18.6772 +- 5.15132,0.80921 +- 0.207335,0.63107 +- 0.173899
brain,45.7914 +- 56.1777,176.887 +- 146.08,458.572 +- 199.344,1626.01 +- 858.324,23.5676 +- 3.26589,17.7934 +- 3.56651,0.896768 +- 0.050685,0.726881 +- 0.144994
pelvis,32.4928 +- 86.9599,106.113 +- 85.2909,777.534 +- 196.074,2803.77 +- 545.858,26.9345 +- 2.82816,21.2778 +- 2.73855,0.876766 +- 0.0486144,0.71202 +- 0.0937712


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,136.346 +- 140.352,176.534 +- 110.666
MSE,1072.94 +- 527.877,3416.89 +- 1321.13
PSNR,23.8193 +- 3.22935,19.384 +- 2.5202
SSIM,0.818647 +- 0.08101,0.668699 +- 0.0748214


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,145.396 +- 135.398,151.875 +- 61.4835,1360.12 +- 499.537,4540.75 +- 1299.49,24.9918 +- 2.29292,20.7277 +- 1.49729,0.804143 +- 0.0658271,0.622502 +- 0.040727
HN,267.666 +- 164.636,294.985 +- 166.617,1662.89 +- 309.179,4287.15 +- 879.153,20.3369 +- 3.54333,17.2681 +- 3.2801,0.729154 +- 0.0749674,0.615758 +- 0.0965054
TH,161.905 +- 107.718,154.143 +- 42.4856,983.677 +- 302.047,3723.31 +- 533.498,24.0779 +- 2.41707,19.6732 +- 1.89889,0.806087 +- 0.0564999,0.663964 +- 0.0455308
brain,45.6432 +- 10.4622,151.156 +- 27.5659,458.081 +- 58.0883,1558.92 +- 144.748,23.5907 +- 1.09345,18.4232 +- 0.966604,0.896871 +- 0.0134715,0.743843 +- 0.0268678
pelvis,31.9384 +- 12.2583,104.188 +- 19.174,768.831 +- 119.854,2780.95 +- 278.347,26.8729 +- 1.33419,21.298 +- 1.32992,0.876867 +- 0.0170696,0.709191 +- 0.0294098


# Pix2Pix


## Abdomen


In [27]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_abdomen_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [28]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_abdomen_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,35.5146 +- 21.7365,116.792 +- 63.4296
MSE,891.223 +- 294.13,2959.32 +- 861.443
PSNR,26.4703 +- 5.36272,21.3532 +- 5.68751
SSIM,0.856118 +- 0.0435502,0.647128 +- 0.0963763


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,35.5146 +- 21.7365,116.792 +- 63.4296,891.223 +- 294.13,2959.32 +- 861.443,26.4703 +- 5.36272,21.3532 +- 5.68751,0.856118 +- 0.0435502,0.647128 +- 0.0963763


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,34.4824 +- 10.077,116.864 +- 35.3118
MSE,885.013 +- 166.129,3003.4 +- 406.657
PSNR,26.7945 +- 2.12079,20.8457 +- 1.94641
SSIM,0.858553 +- 0.0245733,0.64386 +- 0.0490966


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,34.4824 +- 10.077,116.864 +- 35.3118,885.013 +- 166.129,3003.4 +- 406.657,26.7945 +- 2.12079,20.8457 +- 1.94641,0.858553 +- 0.0245733,0.64386 +- 0.0490966


## Brain


In [29]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_brain_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [30]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_brain_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,42.6296 +- 27.4461,157.01 +- 107.273
MSE,529.886 +- 189.786,1901.94 +- 704.052
PSNR,24.1788 +- 3.3485,18.5622 +- 3.48775
SSIM,0.896459 +- 0.0521861,0.733358 +- 0.157734


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
brain,42.6296 +- 27.4461,157.01 +- 107.273,529.886 +- 189.786,1901.94 +- 704.052,24.1788 +- 3.3485,18.5622 +- 3.48775,0.896459 +- 0.0521861,0.733358 +- 0.157734


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,42.5632 +- 9.11883,140.911 +- 24.5451
MSE,528.866 +- 63.9326,1771.32 +- 141.145
PSNR,24.2005 +- 1.13618,19.0888 +- 1.0783
SSIM,0.896607 +- 0.0158708,0.749861 +- 0.0398606


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
brain,42.5632 +- 9.11883,140.911 +- 24.5451,528.866 +- 63.9326,1771.32 +- 141.145,24.2005 +- 1.13618,19.0888 +- 1.0783,0.896607 +- 0.0158708,0.749861 +- 0.0398606


## Pelvis


In [31]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_pelvis_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [32]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_pelvis_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,24.6619 +- 11.5376,85.2358 +- 32.2456
MSE,664.888 +- 161.113,2426.32 +- 585.849
PSNR,27.6317 +- 2.29496,22.3839 +- 2.28337
SSIM,0.89894 +- 0.0251683,0.764302 +- 0.0835479


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
pelvis,24.6619 +- 11.5376,85.2358 +- 32.2456,664.888 +- 161.113,2426.32 +- 585.849,27.6317 +- 2.29496,22.3839 +- 2.28337,0.89894 +- 0.0251683,0.764302 +- 0.0835479


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,24.6217 +- 3.61799,83.8636 +- 14.3398
MSE,661.938 +- 78.5414,2403.13 +- 279.884
PSNR,27.5955 +- 1.06959,22.4484 +- 1.20732
SSIM,0.898667 +- 0.0124314,0.763811 +- 0.0324249


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
pelvis,24.6217 +- 3.61799,83.8636 +- 14.3398,661.938 +- 78.5414,2403.13 +- 279.884,27.5955 +- 1.06959,22.4484 +- 1.20732,0.898667 +- 0.0124314,0.763811 +- 0.0324249


## Thorax


In [33]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_thorax_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [34]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_thorax_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,34.066 +- 14.343,128.719 +- 54.9162
MSE,771.337 +- 279.583,2878.05 +- 826.963
PSNR,27.0535 +- 7.60064,21.298 +- 7.95816
SSIM,0.868913 +- 0.0390757,0.68027 +- 0.0972599


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
TH,34.066 +- 14.343,128.719 +- 54.9162,771.337 +- 279.583,2878.05 +- 826.963,27.0535 +- 7.60064,21.298 +- 7.95816,0.868913 +- 0.0390757,0.68027 +- 0.0972599


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,34.1529 +- 6.9838,133.6 +- 29.1502
MSE,772.92 +- 148.294,3008.38 +- 217.396
PSNR,27.1284 +- 1.85764,19.8217 +- 1.67067
SSIM,0.868744 +- 0.0208395,0.666592 +- 0.0476561


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
TH,34.1529 +- 6.9838,133.6 +- 29.1502,772.92 +- 148.294,3008.38 +- 217.396,27.1284 +- 1.85764,19.8217 +- 1.67067,0.868744 +- 0.0208395,0.666592 +- 0.0476561


## Head and Neck


In [35]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_headneck_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [36]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_headneck_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,58.0889 +- 52.1225,141.686 +- 123.703
MSE,924.771 +- 531.765,2186.61 +- 833.337
PSNR,25.6518 +- 10.0477,21.728 +- 10.4413
SSIM,0.825289 +- 0.0913395,0.698898 +- 0.130104


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
HN,58.0889 +- 52.1225,141.686 +- 123.703,924.771 +- 531.765,2186.61 +- 833.337,25.6518 +- 10.0477,21.728 +- 10.4413,0.825289 +- 0.0913395,0.698898 +- 0.130104


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,61.8388 +- 27.0338,161.135 +- 77.5774
MSE,897.383 +- 164.704,2281.34 +- 232.672
PSNR,25.6742 +- 1.77638,19.8059 +- 1.94035
SSIM,0.821402 +- 0.0366683,0.667739 +- 0.0826987


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
HN,61.8388 +- 27.0338,161.135 +- 77.5774,897.383 +- 164.704,2281.34 +- 232.672,25.6742 +- 1.77638,19.8059 +- 1.94035,0.821402 +- 0.0366683,0.667739 +- 0.0826987


## Fullbody


In [37]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_allregion_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [38]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_allregion_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Slice metrics | non-finite values ignored in stats -> PSNR_masked: 2
Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,41.7938 +- 34.7524,136.056 +- 98.7396
MSE,733.222 +- 372.682,2405.07 +- 924.712
PSNR,25.8779 +- 8.57675,20.7074 +- 8.83241
SSIM,0.867943 +- 0.0647621,0.701047 +- 0.131393


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,35.055 +- 21.6799,112.518 +- 61.046,891.179 +- 315.494,2953.43 +- 893.441,27.2751 +- 9.79461,22.2567 +- 10.0773,0.859186 +- 0.0436734,0.658805 +- 0.0901732
HN,64.3728 +- 62.4649,155.9 +- 148.418,983.457 +- 578.415,2342.44 +- 994.732,25.5318 +- 12.0712,21.6337 +- 12.4969,0.816514 +- 0.097349,0.682964 +- 0.136781
TH,36.0213 +- 15.4132,134.897 +- 60.8917,778.535 +- 289.479,2901.62 +- 851.053,27.4931 +- 12.0773,21.7348 +- 12.2829,0.863964 +- 0.0423807,0.674712 +- 0.106508
brain,43.8641 +- 25.9867,159.73 +- 97.8801,517.577 +- 217.082,1855.2 +- 786.982,23.8769 +- 3.36234,18.2584 +- 3.42243,0.891781 +- 0.0577551,0.722838 +- 0.166238
pelvis,28.3824 +- 18.9812,100.329 +- 80.4681,662.208 +- 158.507,2396.35 +- 528.313,26.6892 +- 2.31971,21.4683 +- 2.52391,0.889723 +- 0.0300076,0.746238 +- 0.0855961


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,43.5263 +- 23.1622,137.221 +- 59.9666
MSE,770.341 +- 212.745,2520.03 +- 550.055
PSNR,26.2454 +- 2.40526,19.9706 +- 1.86181
SSIM,0.861605 +- 0.0382378,0.687322 +- 0.0685984


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,34.2774 +- 10.2654,113.526 +- 34.5038,889.687 +- 175.797,3014.75 +- 366.143,27.6237 +- 2.82547,20.9644 +- 1.8187,0.861168 +- 0.0254955,0.653966 +- 0.0492277
HN,69.1054 +- 34.2521,180.559 +- 98.383,960.991 +- 171.904,2445.48 +- 300.629,25.6306 +- 1.90779,19.3112 +- 2.00788,0.813121 +- 0.0364707,0.650528 +- 0.083066
TH,36.1152 +- 7.58853,139.504 +- 33.724,781.876 +- 156.508,3044.28 +- 283.545,27.5953 +- 2.59543,19.4741 +- 1.73451,0.863925 +- 0.0213469,0.660293 +- 0.0545041
brain,43.8175 +- 6.71016,144.308 +- 16.6779,515.133 +- 64.5071,1726.51 +- 162.532,23.8832 +- 0.912779,18.7546 +- 0.865869,0.89186 +- 0.0133934,0.738155 +- 0.0336808
pelvis,28.6316 +- 5.48226,98.58 +- 16.4061,661.653 +- 82.5421,2385.71 +- 240.435,26.631 +- 0.90845,21.495 +- 0.924415,0.888723 +- 0.0157249,0.741845 +- 0.0302484


# CycleGAN


## Abdomen


In [39]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_abdomen_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [40]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_abdomen_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,61.3489 +- 38.3061,174.745 +- 114.408
MSE,1299.88 +- 502.085,4303.47 +- 1462.1
PSNR,24.1423 +- 9.54502,20.0138 +- 9.52877
SSIM,0.805787 +- 0.0540942,0.530968 +- 0.127972


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,61.3489 +- 38.3061,174.745 +- 114.408,1299.88 +- 502.085,4303.47 +- 1462.1,24.1423 +- 9.54502,20.0138 +- 9.52877,0.805787 +- 0.0540942,0.530968 +- 0.127972


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,58.3615 +- 20.5244,174.472 +- 54.3367
MSE,1317.14 +- 268.296,4467.66 +- 597.608
PSNR,24.6651 +- 4.02407,18.8446 +- 2.09964
SSIM,0.809664 +- 0.032682,0.527928 +- 0.0819799


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,58.3615 +- 20.5244,174.472 +- 54.3367,1317.14 +- 268.296,4467.66 +- 597.608,24.6651 +- 4.02407,18.8446 +- 2.09964,0.809664 +- 0.032682,0.527928 +- 0.0819799


## Brain


In [41]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_brain_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [42]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_brain_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,88.7851 +- 45.4188,301.096 +- 126.009
MSE,535.347 +- 224.47,1812.13 +- 737.072
PSNR,19.3218 +- 3.68977,13.6109 +- 2.68268
SSIM,0.80957 +- 0.0788591,0.517178 +- 0.167235


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
brain,88.7851 +- 45.4188,301.096 +- 126.009,535.347 +- 224.47,1812.13 +- 737.072,19.3218 +- 3.68977,13.6109 +- 2.68268,0.80957 +- 0.0788591,0.517178 +- 0.167235


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,89.1496 +- 21.9109,298.628 +- 58.686
MSE,534.995 +- 78.6223,1761.77 +- 188.454
PSNR,19.2954 +- 1.44903,13.5041 +- 1.42003
SSIM,0.808978 +- 0.0364361,0.500211 +- 0.103808


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
brain,89.1496 +- 21.9109,298.628 +- 58.686,534.995 +- 78.6223,1761.77 +- 188.454,19.2954 +- 1.44903,13.5041 +- 1.42003,0.808978 +- 0.0364361,0.500211 +- 0.103808


## Pelvis


In [43]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_pelvis_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [44]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_pelvis_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,81.8099 +- 23.8409,195.527 +- 88.7502
MSE,1443.98 +- 356.169,5180.7 +- 1113.42
PSNR,19.884 +- 1.91953,17.5001 +- 3.07259
SSIM,0.780998 +- 0.0178051,0.525097 +- 0.072071


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
pelvis,81.8099 +- 23.8409,195.527 +- 88.7502,1443.98 +- 356.169,5180.7 +- 1113.42,19.884 +- 1.91953,17.5001 +- 3.07259,0.780998 +- 0.0178051,0.525097 +- 0.072071


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,83.3963 +- 16.1116,208.604 +- 91.4367
MSE,1407.8 +- 297.994,5029.67 +- 969.391
PSNR,19.7102 +- 1.44886,17.0609 +- 3.10696
SSIM,0.780139 +- 0.0131045,0.529231 +- 0.0438917


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
pelvis,83.3963 +- 16.1116,208.604 +- 91.4367,1407.8 +- 297.994,5029.67 +- 969.391,19.7102 +- 1.44886,17.0609 +- 3.10696,0.780139 +- 0.0131045,0.529231 +- 0.0438917


## Thorax


In [45]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_thorax_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [46]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_thorax_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Slice metrics | non-finite values ignored in stats -> PSNR_masked: 3
Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,57.8027 +- 34.5039,218.93 +- 128.959
MSE,812.338 +- 339.499,3026.88 +- 1004.73
PSNR,24.8779 +- 11.463,18.9409 +- 11.5293
SSIM,0.833283 +- 0.0569193,0.591857 +- 0.156856


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
TH,57.8027 +- 34.5039,218.93 +- 128.959,812.338 +- 339.499,3026.88 +- 1004.73,24.8779 +- 11.463,18.9409 +- 11.5293,0.833283 +- 0.0569193,0.591857 +- 0.156856


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,56.3957 +- 21.4975,223.936 +- 91.9382
MSE,811.5 +- 203.615,3152.5 +- 396.074
PSNR,25.1316 +- 3.99161,16.9334 +- 3.144
SSIM,0.834815 +- 0.0359449,0.578733 +- 0.108016


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
TH,56.3957 +- 21.4975,223.936 +- 91.9382,811.5 +- 203.615,3152.5 +- 396.074,25.1316 +- 3.99161,16.9334 +- 3.144,0.834815 +- 0.0359449,0.578733 +- 0.108016


## Head and Neck


In [47]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_head_neck_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [48]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_head_neck_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,88.7262 +- 67.4557,220.816 +- 165.052
MSE,1147.04 +- 710.173,2684.61 +- 1155.82
PSNR,22.6615 +- 10.2761,18.7035 +- 10.714
SSIM,0.762174 +- 0.0917308,0.570678 +- 0.145435


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
HN,88.7262 +- 67.4557,220.816 +- 165.052,1147.04 +- 710.173,2684.61 +- 1155.82,22.6615 +- 10.2761,18.7035 +- 10.714,0.762174 +- 0.0917308,0.570678 +- 0.145435


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,93.3221 +- 35.2958,243.337 +- 102.645
MSE,1091.83 +- 281.093,2748.09 +- 425.621
PSNR,22.8213 +- 1.72067,17.0877 +- 1.5587
SSIM,0.760801 +- 0.0326645,0.543353 +- 0.0756491


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
HN,93.3221 +- 35.2958,243.337 +- 102.645,1091.83 +- 281.093,2748.09 +- 425.621,22.8213 +- 1.72067,17.0877 +- 1.5587,0.760801 +- 0.0326645,0.543353 +- 0.0756491


## Fullbody


In [49]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_allregions_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [50]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_allregions_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Slice metrics | non-finite values ignored in stats -> PSNR_masked: 3
Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,83.7235 +- 48.3903,263.32 +- 142.385
MSE,902.415 +- 549.204,2911.72 +- 1468.73
PSNR,21.4483 +- 8.41056,16.3445 +- 8.46842
SSIM,0.79496 +- 0.0714773,0.544002 +- 0.146881


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,71.2038 +- 45.2119,232.003 +- 139.015,1173.42 +- 498.518,3836.21 +- 1322.34,23.4601 +- 9.72217,18.3124 +- 9.92294,0.803147 +- 0.0613289,0.521497 +- 0.139869
HN,102.144 +- 67.6649,251.436 +- 164.813,1096.96 +- 754.075,2528.04 +- 1222.44,21.7892 +- 11.3035,17.8856 +- 11.7419,0.743012 +- 0.0947052,0.543446 +- 0.145443
TH,70.7095 +- 39.9966,254.482 +- 146.912,895.461 +- 390.836,3245.71 +- 1077.23,23.8887 +- 11.7026,18.2539 +- 11.6768,0.8181 +- 0.0633248,0.563163 +- 0.173374
brain,100.249 +- 38.1675,315.515 +- 116.943,468.342 +- 217.506,1631.17 +- 836.704,18.2935 +- 2.9337,13.066 +- 1.88121,0.792474 +- 0.0647163,0.52221 +- 0.150404
pelvis,62.6483 +- 33.4505,225.386 +- 130.929,1194.14 +- 398.007,4235.91 +- 1024.11,22.1514 +- 2.58131,16.6547 +- 2.66001,0.819159 +- 0.0380783,0.582515 +- 0.103954


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,82.0531 +- 31.7759,257.282 +- 89.6774
MSE,959.221 +- 357.061,3131.83 +- 1079.73
PSNR,22.1378 +- 3.7248,15.9352 +- 2.80049
SSIM,0.79444 +- 0.0443003,0.537648 +- 0.0850759


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,66.5611 +- 29.1278,223.947 +- 99.3627,1194.12 +- 293.962,4012.54 +- 554.366,24.1069 +- 4.51958,17.3856 +- 3.43723,0.808543 +- 0.0418631,0.522159 +- 0.0991575
HN,105.968 +- 33.8091,271.248 +- 95.8539,1039.41 +- 276.009,2608.42 +- 463.679,22.0358 +- 1.92018,16.1823 +- 1.38974,0.74401 +- 0.0330479,0.521134 +- 0.0598953
TH,68.6537 +- 29.2427,258.209 +- 110.93,902.15 +- 249.769,3435.85 +- 461.91,24.1992 +- 4.67338,16.3088 +- 3.58763,0.820375 +- 0.0424916,0.550936 +- 0.127377
brain,99.9188 +- 12.739,302.034 +- 28.8624,468.539 +- 73.8318,1520.28 +- 170.801,18.3097 +- 0.761675,13.1395 +- 0.563861,0.792574 +- 0.0182405,0.523556 +- 0.0481865
pelvis,63.8499 +- 18.0851,227.871 +- 67.8463,1174.07 +- 267.736,4198.39 +- 579.229,22.06 +- 1.90912,16.6049 +- 1.96479,0.817905 +- 0.0249799,0.574127 +- 0.0617434


# Experiment 2

## 32p99

### CUT

In [51]:

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/2_experiment_cut_synthrad_abdomen_32p99/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)

Slice metrics | non-finite values ignored in stats -> MAE_unmasked: 55, MSE_unmasked: 55, PSNR_unmasked: 57, SSIM_unmasked: 55, MAE_masked: 55, MSE_masked: 55, PSNR_masked: 57, SSIM_masked: 55
Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,117.439 +- 55.8068,117.439 +- 55.8068
MSE,4524.52 +- 1334.81,4524.52 +- 1334.81
PSNR,36.2302 +- 3.28233,36.2302 +- 3.28233
SSIM,0.6273 +- 0.0656255,0.6273 +- 0.0656255


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,117.439 +- 55.8068,117.439 +- 55.8068,4524.52 +- 1334.81,4524.52 +- 1334.81,36.2302 +- 3.28233,36.2302 +- 3.28233,0.6273 +- 0.0656255,0.6273 +- 0.0656255


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,115.336 +- 24.556,117.371 +- 27.2922
MSE,4541 +- 574.347,4605.19 +- 540.02
PSNR,36.2685 +- 0.840621,35.9995 +- 0.552692
SSIM,0.62944 +- 0.0262852,0.627112 +- 0.0266859


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,115.336 +- 24.556,117.371 +- 27.2922,4541 +- 574.347,4605.19 +- 540.02,36.2685 +- 0.840621,35.9995 +- 0.552692,0.62944 +- 0.0262852,0.627112 +- 0.0266859


### CycleGAN

In [52]:

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/2_experiment_cyclegan_abdomen_32p99/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)

Slice metrics | non-finite values ignored in stats -> MAE_unmasked: 55, MSE_unmasked: 55, PSNR_unmasked: 70, SSIM_unmasked: 55, MAE_masked: 55, MSE_masked: 55, PSNR_masked: 70, SSIM_masked: 55
Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,161.627 +- 86.0609,161.627 +- 86.0609
MSE,4485.42 +- 1318.47,4485.42 +- 1318.47
PSNR,35.9838 +- 1.90589,35.9838 +- 1.90589
SSIM,0.60744 +- 0.0851225,0.60744 +- 0.0851225


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,161.627 +- 86.0609,161.627 +- 86.0609,4485.42 +- 1318.47,4485.42 +- 1318.47,35.9838 +- 1.90589,35.9838 +- 1.90589,0.60744 +- 0.0851225,0.60744 +- 0.0851225


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,155.413 +- 38.3167,159.636 +- 40.2211
MSE,4499.62 +- 611.976,4552.17 +- 591.149
PSNR,35.996 +- 0.631637,35.8654 +- 0.519152
SSIM,0.613568 +- 0.0453987,0.607734 +- 0.0449712


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,155.413 +- 38.3167,159.636 +- 40.2211,4499.62 +- 611.976,4552.17 +- 591.149,35.996 +- 0.631637,35.8654 +- 0.519152,0.613568 +- 0.0453987,0.607734 +- 0.0449712


### Pix2pix

In [53]:

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/2_experiment_pix2pix_synthrad_abdomen_32p99/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)



Slice metrics | non-finite values ignored in stats -> MAE_unmasked: 55, MSE_unmasked: 55, PSNR_unmasked: 70, SSIM_unmasked: 55, MAE_masked: 55, MSE_masked: 55, PSNR_masked: 70, SSIM_masked: 55
Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,127.467 +- 59.245,127.467 +- 59.245
MSE,3370.37 +- 936.319,3370.37 +- 936.319
PSNR,37.1583 +- 1.61036,37.1583 +- 1.61036
SSIM,0.621342 +- 0.0878483,0.621342 +- 0.0878483


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,127.467 +- 59.245,127.467 +- 59.245,3370.37 +- 936.319,3370.37 +- 936.319,37.1583 +- 1.61036,37.1583 +- 1.61036,0.621342 +- 0.0878483,0.621342 +- 0.0878483


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,124.527 +- 29.7982,126.296 +- 31.2824
MSE,3352.74 +- 416.288,3380.52 +- 381.853
PSNR,37.2023 +- 0.580932,37.1003 +- 0.488377
SSIM,0.625181 +- 0.0498587,0.622135 +- 0.0475174


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,124.527 +- 29.7982,126.296 +- 31.2824,3352.74 +- 416.288,3380.52 +- 381.853,37.2023 +- 0.580932,37.1003 +- 0.488377,0.625181 +- 0.0498587,0.622135 +- 0.0475174


## Sep Input Layer

### CUT

In [54]:

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/2_experiment_cut_synthrad_abdomen_sep_first_layer/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Slice metrics | non-finite values ignored in stats -> MAE_unmasked: 55, MSE_unmasked: 55, PSNR_unmasked: 83, SSIM_unmasked: 55, MAE_masked: 55, MSE_masked: 55, PSNR_masked: 83, SSIM_masked: 55
Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,181.413 +- 217.377,181.413 +- 217.377
MSE,3944.93 +- 1469.18,3944.93 +- 1469.18
PSNR,36.8834 +- 3.33619,36.8834 +- 3.33619
SSIM,0.587645 +- 0.126163,0.587645 +- 0.126163


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,181.413 +- 217.377,181.413 +- 217.377,3944.93 +- 1469.18,3944.93 +- 1469.18,36.8834 +- 3.33619,36.8834 +- 3.33619,0.587645 +- 0.126163,0.587645 +- 0.126163


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,195.011 +- 226.507,201.499 +- 241.396
MSE,3921.9 +- 1107.72,3967.93 +- 1126.73
PSNR,37.0618 +- 2.13533,36.8689 +- 2.04363
SSIM,0.582841 +- 0.113074,0.578012 +- 0.121299


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,195.011 +- 226.507,201.499 +- 241.396,3921.9 +- 1107.72,3967.93 +- 1126.73,37.0618 +- 2.13533,36.8689 +- 2.04363,0.582841 +- 0.113074,0.578012 +- 0.121299


### CycleGAN

In [55]:

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/2_experiment_cyclegan_abdomen_sep_first_layer/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Slice metrics | non-finite values ignored in stats -> MAE_unmasked: 55, MSE_unmasked: 55, PSNR_unmasked: 141, SSIM_unmasked: 55, MAE_masked: 55, MSE_masked: 55, PSNR_masked: 141, SSIM_masked: 55
Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,228.958 +- 153.319,228.958 +- 153.319
MSE,3868.78 +- 1416.12,3868.78 +- 1416.12
PSNR,36.4858 +- 1.9758,36.4858 +- 1.9758
SSIM,0.495445 +- 0.120165,0.495445 +- 0.120165


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,228.958 +- 153.319,228.958 +- 153.319,3868.78 +- 1416.12,3868.78 +- 1416.12,36.4858 +- 1.9758,36.4858 +- 1.9758,0.495445 +- 0.120165,0.495445 +- 0.120165


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,216.182 +- 135.195,221.772 +- 145.263
MSE,3936.67 +- 1068.88,3981.7 +- 1122.32
PSNR,36.7906 +- 2.23967,36.7894 +- 2.83317
SSIM,0.509984 +- 0.100089,0.503543 +- 0.0999761


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,216.182 +- 135.195,221.772 +- 145.263,3936.67 +- 1068.88,3981.7 +- 1122.32,36.7906 +- 2.23967,36.7894 +- 2.83317,0.509984 +- 0.100089,0.503543 +- 0.0999761


### Pix2Pix

In [56]:

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/2_experiment_pix2pix_synthrad_abdomen_sep_input_layers/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Slice metrics | non-finite values ignored in stats -> MAE_unmasked: 55, MSE_unmasked: 55, PSNR_unmasked: 70, SSIM_unmasked: 55, MAE_masked: 55, MSE_masked: 55, PSNR_masked: 70, SSIM_masked: 55
Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,119.694 +- 62.6803,119.694 +- 62.6803
MSE,3162.15 +- 797.262,3162.15 +- 797.262
PSNR,37.3921 +- 1.53022,37.3921 +- 1.53022
SSIM,0.635347 +- 0.0831995,0.635347 +- 0.0831995


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,119.694 +- 62.6803,119.694 +- 62.6803,3162.15 +- 797.262,3162.15 +- 797.262,37.3921 +- 1.53022,37.3921 +- 1.53022,0.635347 +- 0.0831995,0.635347 +- 0.0831995


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,115.369 +- 33.3811,117.079 +- 34.9056
MSE,3110.78 +- 375.589,3141.06 +- 349.507
PSNR,37.4771 +- 0.568067,37.3789 +- 0.487947
SSIM,0.643568 +- 0.0554665,0.639832 +- 0.0548684


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,115.369 +- 33.3811,117.079 +- 34.9056,3110.78 +- 375.589,3141.06 +- 349.507,37.4771 +- 0.568067,37.3789 +- 0.487947,0.643568 +- 0.0554665,0.639832 +- 0.0548684


## Nyul

### CUT

In [57]:

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/2_experiment_cut_synthrad_abdomen_33nyul/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


FileNotFoundError: CSV not found: /local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/2_experiment_cut_synthrad_abdomen_33nyul/test_50/test_metrics_over_all.csv

### CycleGAN

In [ ]:

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/2_experiment_cyclegan_abdomen_33nyul/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Slice metrics | non-finite values ignored in stats -> MAE: 55, MSE: 55, PSNR: 122, SSIM: 55
Overall metrics (slice-based):


,mean +- std
MAE,1042.68 +- 224.291
MSE,677.143 +- 413.011
PSNR,44.5898 +- 4.43406
SSIM,5.44577e-06 +- 1.00279e-05


Metrics by body part (slice-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
AB,1042.68 +- 224.291,677.143 +- 413.011,44.5898 +- 4.43406,5.44577e-06 +- 1.00279e-05


Volume metrics CSV not found: /local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/2_experiment_cyclegan_abdomen_33nyul/test_50/test_metrics_over_volume.csv
Run: python training/scripts/compute_volume_metrics.py <path/to/test_metrics_over_all.csv>


### Pix2Pix

In [ ]:

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/2_experiment_pix2pix_synthrad_abdomen_33nyul/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Slice metrics | non-finite values ignored in stats -> MAE: 55, MSE: 55, PSNR: 96, SSIM: 55
Overall metrics (slice-based):


,mean +- std
MAE,973.616 +- 106.042
MSE,584.805 +- 304.78
PSNR,45.0819 +- 2.58491
SSIM,-6.7216e-06 +- 4.92028e-05


Metrics by body part (slice-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
AB,973.616 +- 106.042,584.805 +- 304.78,45.0819 +- 2.58491,-6.7216e-06 +- 4.92028e-05


Volume metrics CSV not found: /local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/2_experiment_pix2pix_synthrad_abdomen_33nyul/test_50/test_metrics_over_volume.csv
Run: python training/scripts/compute_volume_metrics.py <path/to/test_metrics_over_all.csv>


## N-Peaks

In [ ]:

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/modelName/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


In [ ]:

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/modelName/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


In [ ]:

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/modelName/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)
